# 16 SLURM submission scripts

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part IV — Working on a cluster</span>
    <span class="bp-meta">Notebook&nbsp;16</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Handing your work to a scheduler: writing a <code>#SBATCH</code> submission
    script, loading its environment from inside, submitting it with
    <code>sbatch</code>, watching the queue, and sweeping a whole dataset with a
    job array.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate, turn on the
# (real) module system pointed at this repo's demo modulefiles, and source the
# thin SLURM mock that stands in for a scheduler. data/ is read-only; every job
# and its slurm-*.out live in a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
# Backgrounded job bodies must not print async "[N] PID"/"Done" notices that
# would arrive after a cell and confuse the kernel (Notebook 15).
set +m
cd "$ROOT"
# The module system pages avail/list through `less` when it sees a terminal; the
# kernel runs in a pty, so force no pager (Notebook 14) and point it at our demo
# modulefiles.
export MODULES_PAGER=cat
for _init in /usr/share/modules/init/bash /etc/profile.d/modules.sh /opt/homebrew/opt/modules/init/bash; do
  [ -r "$_init" ] && { source "$_init"; break; }
done
export MODULEPATH="$ROOT/modulefiles"
module purge >/dev/null 2>&1 || true
# The scheduler stand-in: sbatch/squeue/scancel/sacct/sinfo as shell functions.
source "$ROOT/tools/slurm-mock.sh"
slurm_hard_reset
true

## What this notebook is about

Everything in Notebooks 14 and 15 got you *onto* the cluster and your data *across*
to it. This one is about the thing you actually came for: **running the work** — and
on a shared machine you do not just run it, you **hand it to a scheduler.**

A submission script is the whole idea, and it is nothing new: it is a **bash script**
(Part III) with a short header of **`#SBATCH` directives** that tell the scheduler
what resources you want. You write it, **submit** it, and **check on it**; it runs
*later*, on some compute node, in a **fresh non-login shell** — which is exactly why
the environment lesson from Notebook 14 is about to pay off. The loop is:

> **submit → queue → run → collect.**

:::{admonition} A real scheduler we can stand in for
:class: note
There is no SLURM scheduler inside this page, so the course ships a thin **mock**:
`sbatch` runs your job's body right here — in the background, in a fresh shell — and
writes a real `slurm-<jobid>.out`; `squeue`, `scancel`, `sacct`, and `sinfo` all
behave plausibly. Because `#SBATCH` lines are ordinary **comments to bash**, the
scripts you write here are **exactly** what you would submit on ETH's **Euler** or
any other SLURM cluster.

The one thing the mock cannot reproduce is the **queue wait**: here your job starts
immediately, where on a shared cluster it waits its turn behind hundreds of others.
That wait is the scheduler's whole reason to exist — so picture the line even as we
skip it. (As ever: **no physics** — we load a stand-in "code" and never ask what it
computes.)
:::

## A. Why a scheduler

A cluster is **shared**. Hundreds of people want to run hours-long jobs on the same
few thousand cores, so you cannot simply log in and launch your work — you would
trample everyone else (and they, you). Instead the machine runs a **scheduler**
(SLURM, on Euler and most academic clusters). You describe the job and the resources
it needs; SLURM puts it in a **queue**, and when the resources are free it runs your
job on a **compute node** and saves the output for you to collect.

The machine is carved into **partitions** (named queues, often by time limit).
**`sinfo`** shows them — the menu you are choosing from:

```{command-card} sinfo
```

In [2]:
sinfo

PARTITION    AVAIL  TIMELIMIT  NODES  STATE


normal.4h       up    4:00:00     48  idle


normal.24h      up   24:00:00     96  mix


normal.120h     up  120:00:00     24  alloc


(Real Euler partitions are time-based — `normal.4h`, `normal.24h`, and so on — and
busier; this is the mock's tidy stand-in.) One rule of etiquette follows immediately
from "shared": **never run heavy work on the login node.** The login node is the
shared front desk where everyone types; you *submit* real computation to a compute
node, you do not run it where you land.

## B. Anatomy of a submission script

A submission script has three parts, in order: a **shebang**, a **header of
`#SBATCH` directives**, then the **body** (the actual commands). Here is the
smallest complete one — write it and look at its three parts:

In [3]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [4]:
cat > first-job.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=first
#SBATCH --time=00:05:00
#SBATCH --ntasks=1
#SBATCH --mem-per-cpu=1G

echo "Hello from batch job $SLURM_JOB_ID, running as '$SLURM_JOB_NAME'."
echo "I ran on a compute node, in a shell all my own."
EOF

The `#SBATCH` lines look like comments — and to **bash** they *are* comments, which
is why the script still runs as an ordinary file. To **SLURM** they are resource
requests, read before the job starts. They must come **before any real command**.
Here is the workhorse set:

<div class="bp-card">
  <span class="bp-card-cmd">#SBATCH directives</span> — <span class="bp-card-job">resource requests at the top of the script: comments to bash, instructions to SLURM. (A reference table — Euler-anchored.)</span>
  <table>
    <tr><td>--job-name=NAME</td><td>a label for the job, shown in <code>squeue</code></td></tr>
    <tr><td>--time=HH:MM:SS</td><td>wall-clock limit; the job is KILLED when it is exceeded</td></tr>
    <tr><td>--ntasks=N</td><td>number of tasks (e.g. MPI ranks); <code>1</code> for a serial job</td></tr>
    <tr><td>--cpus-per-task=N</td><td>cores per task — for a threaded / OpenMP code</td></tr>
    <tr><td>--nodes=N</td><td>how many compute nodes to spread across</td></tr>
    <tr><td>--mem-per-cpu=SIZE</td><td>memory per core, e.g. <code>2G</code>; too low and the job is OOM-killed</td></tr>
    <tr><td>--partition=NAME</td><td>which queue (from <code>sinfo</code>) to submit to</td></tr>
    <tr><td>--output=FILE</td><td>where output goes (default <code>slurm-%j.out</code>; <code>%j</code> = the job id)</td></tr>
    <tr><td>--array=A-B</td><td>submit a job array — many tasks from one script (§E)</td></tr>
  </table>
</div>

:::{admonition} ⚠ Request realistically — too little is as bad as too much
:class: warning
Two directives bite hardest. If **`--time`** is shorter than the job needs, SLURM
**kills it** the moment the limit passes — hours of work, gone at 99%. If
**`--mem-per-cpu`** is lower than the job needs, it is **OOM-killed**. (And asking
for *too much* of either just makes you wait longer in the queue, since SLURM must
find that much free — and on many clusters it bills your allocation.) Estimate from
a real run, then add a margin.
:::

Now **submit** it. `sbatch` hands the script to the scheduler and prints a **job
ID** — the number you will use to track it:

```{command-card} sbatch
```

In [5]:
sbatch first-job.sh

[1] 4589


Submitted batch job 1000


In [6]:
wait   # mock: let the backgrounded job body finish before we read its output

The job ran on a "compute node", and — this is the part that surprises everyone the
first time — **its output did not come back to your screen.** It went to a file,
`slurm-<jobid>.out`, in the directory you submitted from. That is where results live;
go and find it:

In [7]:
ls slurm-*.out

slurm-1000.out


In [8]:
cat slurm-*.out

Hello from batch job 1000, running as 'first'.


I ran on a compute node, in a shell all my own.


There is the whole loop in miniature: you wrote a script, `sbatch` queued it, it ran
elsewhere, and it left its output in a file for you to collect.

## C. The environment inside the job

Here is the single most important practical lesson in this notebook, and it follows
straight from Notebook 14. Your job runs in a **fresh, non-login shell** on a compute
node. That shell does **not** run your `~/.bashrc`, and it does **not** inherit the
modules you loaded by hand on the login node. So if your script just *calls* a tool,
expecting it to be on `PATH` the way it was when you tested by hand — it will not be.

Watch it fail. First, load our stand-in code **here, in this interactive shell**, and
confirm it is available:

In [9]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [10]:
module load democode/1.0

In [11]:
command -v democode

/home/runner/work/bash-primer/bash-primer/opt/democode-1.0/bin/democode


It is right there on our `PATH`. Now submit a job that simply runs it — **without**
loading it inside the script:

In [12]:
cat > no-module.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=no-module
democode
EOF

In [13]:
sbatch no-module.sh

[1] 4628


Submitted batch job 1001


In [14]:
wait

In [15]:
cat slurm-*.out

no-module.sh: line 3: democode: command not found


**`command not found`** — even though `democode` was loaded right here when we
submitted. The job's fresh shell never saw it. The fix is the whole lesson: **load
the environment *inside* the script**, so the job sets itself up no matter what your
login shell happened to have:

In [16]:
cat > with-module.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=with-module
module load democode/1.0
democode
EOF

In [17]:
rm -f slurm-*.out   # clear the failed job's output so the next listing is clean

In [18]:
sbatch with-module.sh

[1] 4659


Submitted batch job 1002


In [19]:
wait

In [20]:
cat slurm-*.out

democode 1.0 — stand-in simulation code (loaded via the module system)


Same job, one line added, and now it works. **Every** submission script should set up
its own environment — `module load` (and `export` what you need) — in its body. It is
the difference between "it worked when I ran it by hand" and a job that actually runs.

In [21]:
module purge >/dev/null 2>&1 || true   # leave the interactive shell clean again

## D. Submit and monitor

You rarely submit one job and walk away. The real rhythm is **submit, then watch**:
is it queued, running, done? Three commands cover it. **`squeue`** shows the queue,
**`scancel`** cancels a job, and **`sacct`** reports on jobs that have already
finished.

```{command-card} squeue
```

```{command-card} scancel
```

```{command-card} sacct
```

Submit a longer job and watch it move through the system. First, there it is in the
queue — running (`R`), with the job id, name, and partition:

In [22]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset
cat > long-job.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=long-job
#SBATCH --time=04:00:00
echo "starting the long run..."
sleep 300
echo "done"
EOF

In [23]:
jobid=$(sbatch long-job.sh | awk '{print $NF}')
echo "submitted, tracking job $jobid"

submitted, tracking job 1003


In [24]:
squeue -u "$USER"

             JOBID   PARTITION       NAME      USER ST     TIME  NODES NODELIST


              1003      normal   long-job    runner  R     0:01      1 compute-01


On a real cluster you would `watch squeue` (Notebook 15) and wait for it to finish.
Here, suppose you spot a mistake and want it gone — **`scancel`** it by id:

In [25]:
scancel "$jobid"

In [26]:
squeue -u "$USER"

             JOBID   PARTITION       NAME      USER ST     TIME  NODES NODELIST


The queue is empty again. And after the fact — whether a job finished or was
cancelled — **`sacct`** is the history book that says how it ended:

In [27]:
sacct

       JobID      JobName       State ExitCode


------------ ------------ ----------- --------


        1003     long-job   CANCELLED      0:0


`CANCELLED`, as expected. (A finished job would read `COMPLETED`; a job that ran past
its `--time` reads `TIMEOUT`; one that overran its memory, `OUT_OF_MEMORY`.) The
loop, then, is: **`sbatch` → `squeue` to watch → read `slurm-<jobid>.out` for the
result, or `sacct` for the verdict.**

## E. Job arrays — one script, many tasks

The cluster's real power is doing the *same* work over a *whole dataset* at once. You
have 200 trajectories to analyze; you do not write 200 scripts, and you do not loop
200 `sbatch`es. You write **one** script and submit it as a **job array**.

`#SBATCH --array=1-N` tells SLURM to run the script **N times**. Each run — each
**task** — is identical except for one variable, **`$SLURM_ARRAY_TASK_ID`**, which is
its index (1, 2, 3, …). You use that index to pick *which* item this task handles —
exactly the indexing logic of a loop (Notebook 13), but the scheduler runs the tasks
in parallel for you. Here is a sweep over a small list of trajectory files:

In [28]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset
# Three stand-in trajectory files; an .xyz file's first line is its atom count.
printf '12\nwater trimer\n' > traj_01.xyz
printf '38\nlj cluster\n'   > traj_02.xyz
printf '7\nfragment\n'      > traj_03.xyz
ls traj_*.xyz > files.txt

We have a list of files (one per line) and a script whose task picks the line
matching its array index, then "analyzes" it (here: reports the atom count):

In [29]:
cat files.txt

traj_01.xyz


traj_02.xyz


traj_03.xyz


In [30]:
cat > sweep.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=sweep
#SBATCH --array=1-3
#SBATCH --time=00:10:00

module load democode/1.0                       # set up the environment (§C)
file=$(sed -n "${SLURM_ARRAY_TASK_ID}p" files.txt)   # this task's file
atoms=$(head -n 1 "$file")                      # the "analysis"
echo "task ${SLURM_ARRAY_TASK_ID}: ${file} has ${atoms} atoms"
EOF

Submit it **once**; SLURM expands it into three tasks:

In [31]:
sbatch sweep.sh

[1] 4769


Submitted batch job 1004


In [32]:
wait

Each task wrote its own output file, `slurm-<jobid>_<task>.out`:

In [33]:
ls slurm-*_*.out

slurm-1004_1.out  slurm-1004_2.out  slurm-1004_3.out


In [34]:
cat slurm-*_*.out

task 1: traj_01.xyz has 12 atoms


task 2: traj_02.xyz has 38 atoms


task 3: traj_03.xyz has 7 atoms


Three files processed from one submission, each by its own task. Collecting the
per-task outputs into a single result is the everyday last step:

In [35]:
cat slurm-*_*.out | sort > results.txt; cat results.txt

task 1: traj_01.xyz has 12 atoms


task 2: traj_02.xyz has 38 atoms


task 3: traj_03.xyz has 7 atoms


The variables SLURM sets inside the job (you read them, you never set them):

<div class="bp-card">
  <span class="bp-card-cmd">$SLURM_* variables</span> — <span class="bp-card-job">set by the scheduler inside the running job; read them in your script. (A reference table.)</span>
  <table>
    <tr><td>$SLURM_JOB_ID</td><td>the job's id (the number <code>sbatch</code> returned)</td></tr>
    <tr><td>$SLURM_JOB_NAME</td><td>the <code>--job-name</code> you gave it</td></tr>
    <tr><td>$SLURM_SUBMIT_DIR</td><td>the directory you ran <code>sbatch</code> from</td></tr>
    <tr><td>$SLURM_ARRAY_TASK_ID</td><td>which array task this is — the index you sweep on</td></tr>
    <tr><td>$SLURM_CPUS_PER_TASK</td><td>the <code>--cpus-per-task</code> you requested</td></tr>
  </table>
</div>

## Exercises

The runnable ones submit through the mock: each lives in a fresh `scratch/`, every
`slurm-*.out` is cleaned up between them, and `data/` stays read-only. Job IDs vary,
so the checks read **content**, not numbers. (When a hidden step needs the
background job to finish before reading its output, it `wait`s — the mock's stand-in
for "poll `squeue` until it is done".)

### Warm-up 1 (worked) — Anatomy and submit

Write a minimal script (shebang + a couple of `#SBATCH` lines + an `echo` body),
`sbatch` it, find its `slurm-<jobid>.out`, and read it.

In [36]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [37]:
# (solution hidden on the public site)


[1] 4825


Submitted batch job 1005


--- output file: ---


slurm-1005.out


--- its contents: ---


hello from job 1005 on the cluster


In [38]:
check 'ls slurm-*.out >/dev/null 2>&1 && grep -q "hello from job" slurm-*.out' \
      "the script was submitted and its output landed in slurm-<jobid>.out"

✓ the script was submitted and its output landed in slurm-<jobid>.out


### Warm-up 2 (your turn) — Submit and monitor

Submit a job, see it in `squeue`, then `scancel` it and confirm with `sacct` that it
is recorded as cancelled.

In [39]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset
cat > job.sh <<'EOF'
#!/usr/bin/env bash
#SBATCH --job-name=monitor-me
echo "working..."
sleep 300
EOF

In [40]:
# (solution hidden on the public site)


submitted job 1006


             JOBID   PARTITION       NAME      USER ST     TIME  NODES NODELIST


              1006      normal monitor-me    runner  R     0:01      1 compute-01


--- queue after cancel: ---


             JOBID   PARTITION       NAME      USER ST     TIME  NODES NODELIST


--- history: ---


       JobID      JobName       State ExitCode


------------ ------------ ----------- --------


        1006   monitor-me   CANCELLED      0:0


In [41]:
check 'sacct | grep -qE "CANCELLED|COMPLETED"' \
      "the job appeared in the queue and its end-state was recorded by sacct"

✓ the job appeared in the queue and its end-state was recorded by sacct


### Applied 1 (your turn) — The environment inside the job

Submit a script that runs `democode` **without** loading its module (it fails), then
one that `module load`s it **inside** (it works) — the §C lesson, in your hands.

In [42]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [43]:
# (solution hidden on the public site)


[1] 4940


Submitted batch job 1007


--- without module load: ---


bare.sh: line 3: democode: command not found


[1] 4971


Submitted batch job 1008


--- with module load inside: ---


democode 1.0 — stand-in simulation code (loaded via the module system)


In [44]:
# Re-run both in isolation so the check sees each result on its own.
cd "$ROOT/scratch"; rm -f slurm-*.out
sbatch bare.sh; wait; bare_out="$(cat slurm-*.out)"; rm -f slurm-*.out
sbatch fixed.sh; wait; fixed_out="$(cat slurm-*.out)"
check 'echo "$bare_out" | grep -q "not found" && echo "$fixed_out" | grep -q "democode 1.0"' \
      "the job failed without the module load and succeeded with it loaded inside"

[1] 5002


Submitted batch job 1009


[1] 5032


Submitted batch job 1010


✓ the job failed without the module load and succeeded with it loaded inside


### Applied 2 (worked) — A real Euler submission script

Write a properly-formed script with the full workhorse header (`--partition`,
`--time`, `--ntasks`, `--cpus-per-task`, `--output`), set up its environment, and
submit it. Then "lint" it — confirm the directives are present.

In [45]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [46]:
# (solution hidden on the public site)


--- the #SBATCH header: ---


#SBATCH --job-name=production


#SBATCH --partition=normal.4h


#SBATCH --time=02:00:00


#SBATCH --ntasks=1


...

[1] 5076


Submitted batch job 1011


--- output (note the custom --output name): ---


running production job 1011 on 1 core(s)


democode 1.0 — stand-in simulation code (loaded via the module system)


In [47]:
check 'grep -q -- "--partition" euler-run.sh && grep -q -- "--time" euler-run.sh && grep -q -- "--cpus-per-task" euler-run.sh && ls run-*.out >/dev/null 2>&1 && grep -q "democode 1.0" run-*.out' \
      "the script has a well-formed Euler header, honoured --output, and ran"

✓ the script has a well-formed Euler header, honoured --output, and ran


### Composite — putting it together (a job-array sweep)

The realistic "run my analysis over the whole dataset" workflow, composing the file
list (Notebook 5), array indexing (Notebook 13), the script (Notebook 12), and the
environment inside (Notebook 14). Write **one** array script that processes **one
trajectory per task** — `$SLURM_ARRAY_TASK_ID` indexing a file list — `module load`s
inside, writes each task's result, then collect the results into a table.

In [48]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset
printf '12\n' > frame_01.xyz
printf '38\n' > frame_02.xyz
printf '54\n' > frame_03.xyz
printf '9\n'  > frame_04.xyz
ls frame_*.xyz > frames.txt

In [49]:
# (solution hidden on the public site)


[1] 5123


Submitted batch job 1012


--- per-task outputs: ---


slurm-1012_1.out  slurm-1012_2.out  slurm-1012_3.out  slurm-1012_4.out


--- collected results table: ---


frame_01.xyz 12


frame_02.xyz 38


frame_03.xyz 54


frame_04.xyz 9


In [50]:
n=$(ls "$ROOT"/scratch/slurm-*_*.out 2>/dev/null | wc -l | tr -d ' ')
check '[ "$n" = "4" ] && grep -q "frame_02.xyz 38" "$ROOT/scratch/results.txt" && [ "$(wc -l < "$ROOT/scratch/results.txt")" -eq 4 ]' \
      "the array ran four tasks, one per file, and the collected table has all four results"

✓ the array ran four tasks, one per file, and the collected table has all four results


### Optional stretch (conceptual) — Generate a script, and the resource question

No grade. Two directions. **(a)** A submission script is just text, so you can
**generate** one with a heredoc (Notebook 12) — handy when a parameter changes per
run:

In [51]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; cd scratch; slurm_reset

In [52]:
# (solution hidden on the public site)


generated:


run-1.sh  run-2.sh  run-4.sh


--- run-4.sh: ---


#!/usr/bin/env bash


#SBATCH --job-name=scaling-4


#SBATCH --cpus-per-task=4


#SBATCH --time=00:30:00


echo "would run on 4 core(s)"


**(b)** Every script in this notebook simply *guessed* at `--ntasks` and
`--cpus-per-task`. How many should you actually request? Asking for more cores does
not automatically run faster — and wastes your allocation while you wait longer in
the queue. That question is the whole of Notebook 17. (And for an *interactive*
session on a compute node — a shell to test in, rather than a batch job — the tool is
`srun`; read your cluster's docs for it.)

## Outlook

You can now write a real submission script, set up its environment from the inside,
submit and monitor it with `sbatch`/`squeue`/`scancel`/`sacct`, and sweep an entire
dataset with a single job array. That is the working cluster loop, end to end.

But every script you wrote **guessed** at `--ntasks` and `--cpus-per-task`. Ask for
too few and the job crawls; ask for too many and you wait longer in the queue, waste
your allocation, and — the part that surprises people — often do not run any faster
at all. **How many should you actually request?** Next (Notebook 17): read real
scaling data, compute parallel efficiency, and *justify* the number — the reasoning
behind a real CSCS or LUMI allocation proposal.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box to write and
    submit your own scripts against the SLURM mock — and, if you have access to a real
    cluster (ETH's <b>Euler</b>, say), the very same scripts work there unchanged. The
    published notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions — to teach from or to check your own work — get in touch:
    <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>